
**🤖 AI Lab Partner Policy: STRICTLY Opt-In Code Generation**

In this course, we treat AI tools (like ChatGPT, Gemini, Copilot) as **Lab Partners**, not solution generators. You must use the following prompt to ensure the AI acts responsibly.

**1. Copy the text inside the block below**
**2. Open your AI Assistant (Gemini, ChatGPT, etc.)**
**3. Paste the text to set the rules for the session**

> "I am a student in an Intro to Machine Learning course. Please act as my **ML Lab Partner**.
> 
> **Your Rules:**
> 
> 1. **Code Generation is STRICTLY Opt-In:** You **MUST NOT** generate any runnable Python code unless my message starts with one of the specific prefixes below (`code:` or `output:`).
>    * *Default Behavior:* If I ask 'How do I...?' or 'Help me with...', explain the strategy in English, provide pseudocode, or use illustrative examples. Do not generate runnable solution code.
> 
> 2. **The 'code:' Trigger (Logic & Calculation):** 
>    * When generating code, prioritize simplicity and human readability. Avoid complex syntax.
>    * **Constraint:** When I use this trigger, provide **only one single line of code**. Do not write full blocks.
> 
> 3. **The 'output:' Trigger (Formatting & Printing):**
>    * Use this ONLY when I request code to print results, format tables, or create plots.
>    * **Exception:** For this trigger only, you **MAY** provide full multi-line code blocks to handle the verbose syntax of formatting or plotting.
> 
> 4. **Wait for Me:** After providing the code, stop immediately. Wait for me to run it and ask for the next step.
> 
> 5. **Explain Briefly:** Add a short comment explaining what the code does.
> 
> 6. **Catch Logic Errors:** If I ask for a step that is methodologically wrong (like testing on training data), stop me and explain the error before proceeding."


# Lecture 19: Word Embeddings --- Meaning as Vectors

Last time, we built Markov text generators that capture local word patterns. But they don't understand *meaning*. Today, we explore **word embeddings**: representing words as vectors where similar meanings are close together.

## Setup

We'll use a pre-trained **GloVe** model trained on Wikipedia and Gigaword (news). Each word is represented as a 100-dimensional vector.

**Note:** The model file is ~130 MB and downloads on first use. All words are lowercase.

In [ ]:
import sys, subprocess
try:
    import gensim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gensim'])

import numpy as np

# Load the pre-trained GloVe model
import gensim.downloader as api
print("Loading GloVe model (downloads ~130 MB on first run)...")
model = api.load('glove-wiki-gigaword-100')
print(f"Model loaded! Vocabulary size: {len(model)} words")
print(f"Each word is a vector of {model.vector_size} numbers")

## 1. Words as Vectors

Each word in the model is represented as a vector of 300 numbers. Let's look at what this means.

In [ ]:
# Look at the vector for one word
word = 'king'
vector = model[word]
print(f"Vector for '{word}': {vector[:10]}...")  # Show first 10 of 100 numbers
print(f"Shape: {vector.shape}")
print(f"Length (norm): {np.linalg.norm(vector):.2f}")

Each number represents something about the word's meaning --- but unlike our toy example with "royal", "gender", and "age" dimensions, the 100 dimensions don't have simple human-readable labels. The computer figured out its own useful dimensions from reading text.

## 2. Finding Similar Words

If two words have similar meanings, their vectors should be close together. We can find the most similar words to any word by searching for the nearest vectors.

In [ ]:
# Find words most similar to a given query
query = 'soccer'
print(f"Most similar to '{query}':")
for word, score in model.most_similar(query):
    print(f"  {word}: {score:.3f}")

**TODO:** Pick 3--4 words from *different* categories (e.g., a food, a profession, an emotion, an animal) and use `model.most_similar()` to find the 5 nearest neighbors for each. Print the results in the same format as the cell above. Do the neighbors make sense? Do any of them surprise you?

In [ ]:
# YOUR CODE HERE
# TODO: Choose your own query words (e.g., a specific food, animal, emotion, sport)
my_queries = [...]

# TODO: For each word in my_queries, find its top 5 similar words and print them.
# Hint: You can use model.most_similar(..., topn=5)
for query in my_queries:
    ...
raise NotImplementedError()

**Answer:**

## 3. Analogies: Vector Arithmetic with Meaning

The most famous property of word embeddings is that you can do **arithmetic** with meaning.

The classic example: **king - man + woman = queen**

The idea: the vector from "man" to "king" captures the concept of "royalty". Adding that same direction to "woman" gives us "queen".

In [ ]:
# Solve analogy: base - man + woman = ?
base = 'king'
result = model.most_similar(positive=[base, 'woman'], negative=['man'], topn=3)
print(f"{base} - man + woman = ?")
for word, score in result:
    print(f"  {word}: {score:.3f}")

**TODO:** Come up with 3--4 analogies of your own to test the model (e.g., capitals, slow:slower, verb tenses). Define them using the format we used above (giving positive and negative words) and use `model.most_similar()` to solve them.

Remember the format is: `result = model.most_similar(positive=[...], negative=[...], topn=...)`

In [ ]:
# YOUR CODE HERE
# TODO: Define your own analogies.
my_analogies = [
    ...
]
#
# TODO: Loop through your analogies and solve them using model.most_similar()
# for ... in my_analogies:
#     ...
raise NotImplementedError()

**Answer:**

## 4. Combining Concepts

We can also **add** word vectors to combine concepts and see what the model finds.

In [ ]:
# Combine two concepts. For each pair we show the top-5 matches
# and also the similarity score for a few words we might *expect* to see.
combinations = [
    (['spain', 'sports'],        ['soccer', 'tennis', 'basketball']),
    (['spanish', 'food'],        ['paella', 'tapas', 'wine']),
    (['japanese', 'cuisine'],    ['sushi', 'rice', 'noodle']),
    (['computer', 'music'],      ['synthesizer', 'recording', 'digital']),
]

for words, expected in combinations:
    # Normalized sum of the two word vectors
    combined = model[words[0]] + model[words[1]]
    combined = combined / np.linalg.norm(combined)

    # Top matches, excluding the input words themselves
    results = model.similar_by_vector(combined, topn=12)
    filtered = [(w, s) for w, s in results if w.lower() not in [x.lower() for x in words]][:8]
    top_str = ', '.join(f"{w} ({s:.2f})" for w, s in filtered)

    # Similarity for specific expected words
    exp_scores = []
    for e in expected:
        e_vec = model[e] / np.linalg.norm(model[e])
        sim = float(np.dot(combined, e_vec))
        exp_scores.append(f"{e} ({sim:.2f})")
    exp_str = ', '.join(exp_scores)

    print(f"{' + '.join(words)}:")
    print(f"  Top matches: {top_str}")
    print(f"  Expected:    {exp_str}")
    print()

**Question:** Compare the results for `japanese + cuisine` and `spanish + food`. The Japanese pairing finds actual dishes (sushi, rice, noodle) with decent scores, while the Spanish pairing returns mostly generic commodity words (foods, supplies, products). What two factors explain this difference?

**Answer:**

## 5. Bias in Embeddings

Word embeddings learn from real text, which contains real **biases**. Let's see an example.

In [ ]:
# For each profession: profession - man + woman = ?
for profession in ['doctor', 'engineer']:
    result = model.most_similar(positive=[profession, 'woman'], negative=['man'], topn=5)
    print(f"{profession} - man + woman = ?")
    for word, score in result:
        print(f"  {word}: {score:.3f}")
    print()

The model often maps male professions to female professions rather than keeping the same profession. This reflects **gender stereotypes** in the training data (Google News articles).

This is an important lesson: **ML models learn patterns from data, including harmful biases.** We'll discuss this more in L21.

**TODO:** Further Exploration! Can you find 2--3 other words (e.g., other professions, adjectives like 'smart' or 'beautiful') that exhibit gender bias in the same way? Use the code pattern from above to test them.

In [ ]:
# YOUR CODE HERE
# TODO: Test 2-3 of your own words for gender bias
test_words = [...]

# for word in test_words:
#     ...
raise NotImplementedError()

## 6. The Limitation: One Word, One Vector

Word embeddings give each word a single, fixed vector. But many words have multiple meanings depending on context.

Think about the word "pound":
- "She paid 20 **pounds** for the ticket." (British currency)
- "The baby weighs 8 **pounds**." (unit of weight)
- "He will **pound** the nail into the wall." (verb: to hit)

In our model, "pound" has just one vector --- a blend of all its meanings.

In [ ]:
# A polysemous word — "pound" has multiple meanings blended into one vector
query = 'pound'
print(f"Most similar to '{query}':")
for word, score in model.most_similar(query, topn=12):
    print(f"  {word}: {score:.3f}")

You'll see a mix of **currency** words (*dollar*, *euro*, *sterling*, *pence*, *franc*) and **weight/unit** words (*kilogram*, *ounce*, *barrel*). The single vector is a compromise that doesn't fully capture any one meaning.

**The solution?** We need representations that **change based on context** --- different vectors for "pound" depending on the surrounding words. That's exactly what **attention mechanisms** and **transformers** do, which we'll cover next time.

**TODO:** Further Exploration! Find 2--3 other polysemous words (words with multiple meanings, e.g., 'bank', 'bark', 'bat', 'apple') and print their top 10-12 similar words. Do you see the different meanings blended together?

In [ ]:
# YOUR CODE HERE
# TODO: Test 2-3 other polysemous words
poly_words = [...]

# for word in poly_words:
#     ...
raise NotImplementedError()

## Summary

1. **Word embeddings** map words to vectors (100 numbers in this GloVe model)
2. Learned from billions of words: similar contexts → similar vectors
3. **Similarity**: nearest vectors to "soccer" = other sports
4. **Analogies**: king - man + woman ≈ queen (vector arithmetic captures relationships)
5. **Composition**: adding vectors combines concepts (spain + sports → soccer, sporting)
6. **Bias warning**: embeddings reflect biases in training data
7. **Limitation**: one fixed vector per word, but meaning depends on context
8. **Next (L21)**: attention & transformers --- context-dependent representations